# AI RAG Assistant Using LangChain
## Quest Analytics – Research Document Intelligence

This notebook covers all **6 tasks** for the final project:

| Task | Topic | Screenshot File |
|------|-------|-----------------|
| 1 | PDF Document Loader | `pdf_loader.png` |
| 2 | Text Splitter | `code_splitter.png` |
| 3 | Embeddings (HuggingFace) | `embedding.png` |
| 4 | Chroma Vector Database | `vectordb.png` |
| 5 | Retriever | `retriever.png` |
| 6 | Gradio QA Bot | `QA_bot.png` |

---
### Prerequisites
1. Run `pip install -r requirements.txt`
2. Copy `.env.example` → `.env` and paste your **FREE** HuggingFace token
   - Get a free token at: https://huggingface.co/settings/tokens  (takes 2 min)
3. Download the sample PDF: https://arxiv.org/pdf/1706.03762  → save as `sample_paper.pdf`

## Setup – Load Environment Variables

Put your **free** HuggingFace token in a `.env` file (see `.env.example`).  
`HUGGINGFACEHUB_API_TOKEN=hf_...`  

No IBM account, no credit card — completely free!

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads HUGGINGFACEHUB_API_TOKEN from .env file

# ─── Your free HuggingFace token ─────────────────────────────────────────────
# Get it free at: https://huggingface.co/settings/tokens
HF_TOKEN = os.getenv('HUGGINGFACEHUB_API_TOKEN', '')

if HF_TOKEN:
    print('✅  HuggingFace token loaded successfully from .env')
    print(f'   Token preview: {HF_TOKEN[:8]}...')
else:
    print('⚠️  No token found in .env')
    print('   → Get a free token at: https://huggingface.co/settings/tokens')
    print('   → Add it to .env:  HUGGINGFACEHUB_API_TOKEN=hf_...')


---
## Task 1 – Load Document Using LangChain (PDF Loader)
> **Screenshot:** Save the output of this cell as **`pdf_loader.png`**

We use `PyPDFLoader` from LangChain Community to load a PDF research paper.  
Each page becomes a `Document` object with `.page_content` and `.metadata`.

In [ ]:
# ─── Task 1: Load Document Using LangChain ───────────────────────────────────
from langchain_community.document_loaders import PyPDFLoader

# Path to the PDF – update this path if needed.
# Download the sample paper at:
#   https://arxiv.org/pdf/1706.03762   ("Attention Is All You Need")
PDF_PATH = "sample_paper.pdf"

loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

print(f"✅  PDF loaded successfully")
print(f"   Total pages  : {len(pages)}")
print(f"   Document type: {type(pages[0]).__name__}")
print()
print("── First page metadata ──")
print(pages[0].metadata)
print()
print("── First 500 characters of page 1 ──")
print(pages[0].page_content[:500])

---
## Task 2 – Apply Text Splitting Techniques
> **Screenshot:** Save the output of this cell as **`code_splitter.png`**

We use `RecursiveCharacterTextSplitter` to break the document into smaller, overlapping chunks.  
Overlap ensures context is not lost at chunk boundaries.

In [ ]:
# ─── Task 2: Apply Text Splitting Techniques ──────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,   # maximum characters per chunk
    chunk_overlap = 200,    # characters shared between adjacent chunks
    length_function = len,
    separators    = ["\n\n", "\n", " ", ""]
)

splits = text_splitter.split_documents(pages)

print(f"✅  Text splitting complete")
print(f"   Original pages : {len(pages)}")
print(f"   Total chunks   : {len(splits)}")
print(f"   Chunk size     : 1000 chars  |  Overlap: 200 chars")
print()
print("── Sample chunk (index 2) ──")
print(f"Content ({len(splits[2].page_content)} chars):")
print(splits[2].page_content)
print()
print("Metadata:", splits[2].metadata)

---
## Task 3 – Embed Documents Using HuggingFace Embeddings
> **Screenshot:** Save the output of this cell as **`embedding.png`**

We use `HuggingFaceEmbeddings` with the `sentence-transformers/all-MiniLM-L6-v2` model.  
This runs **100% locally on CPU** — no API key required, completely free.  
Each text chunk is converted into a 384-dimensional dense vector.

In [ ]:
# ─── Task 3: Embed Documents Using HuggingFace Embeddings ────────────────────
from langchain_huggingface import HuggingFaceEmbeddings

# Runs entirely locally on CPU – no API key needed!
embeddings = HuggingFaceEmbeddings(
    model_name    = "sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True},
)

# Test embedding with the first chunk
sample_text   = splits[0].page_content
sample_vector = embeddings.embed_query(sample_text)

print(f"✅  Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2")
print(f"   Runs locally – FREE, no API key required")
print(f"   Embedding dimensions : {len(sample_vector)}")
print(f"   Sample text (first 100 chars): {sample_text[:100]}...")
print()
print(f"   First 10 embedding values:")
print([round(v, 6) for v in sample_vector[:10]])


---
## Task 4 – Create and Configure Chroma Vector Database
> **Screenshot:** Save the output of this cell as **`vectordb.png`**

We store all embedded document chunks inside a **Chroma** vector database.  
ChromaDB enables fast similarity search over dense vectors.

In [ ]:
# ─── Task 4: Create Chroma Vector Database ────────────────────────────────────
from langchain_community.vectorstores import Chroma

PERSIST_DIR = "./chroma_db"

vectordb = Chroma.from_documents(
    documents        = splits,
    embedding        = embeddings,
    persist_directory= PERSIST_DIR,
)

print(f"✅  Chroma vector database created")
print(f"   Persist directory   : {PERSIST_DIR}")
print(f"   Documents indexed   : {vectordb._collection.count()}")
print()

# Quick similarity search test
test_query = "What is the main topic of this paper?"
results    = vectordb.similarity_search(test_query, k=2)

print(f"── Similarity search test (query: '{test_query}') ──")
for i, doc in enumerate(results):
    print(f"\nResult {i+1} (page {doc.metadata.get('page', 'N/A')}):")
    print(doc.page_content[:300])

---
## Task 5 – Develop a Retriever to Fetch Document Segments
> **Screenshot:** Save the output of this cell as **`retriever.png`**

We expose Chroma as a LangChain **Retriever**.  
The retriever fetches the top-k most relevant chunks for any natural language query.

In [ ]:
# ─── Task 5: Develop a Retriever ──────────────────────────────────────────────
retriever = vectordb.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": 3}         # return top-3 relevant chunks
)

query = "What this paper is talking about?"
docs  = retriever.invoke(query)

print(f"✅  Retriever configured (ChromaDB | similarity search | top-3)")
print(f"   Query  : {query}")
print(f"   Chunks retrieved : {len(docs)}")
print()

for i, doc in enumerate(docs):
    print(f"{'─'*60}")
    print(f"Chunk {i+1}  |  Source page: {doc.metadata.get('page', 'N/A')}")
    print(doc.page_content[:400])
    print()


---
## Task 6 – Construct a QA Bot (LangChain + LLM + Gradio)
> **Screenshot:** Save the Gradio UI as **`QA_bot.png`**  
> Make sure the screenshot shows a **PDF uploaded** and the **query/answer** visible.

We wire everything together:
- **LLM**: `mistralai/Mistral-7B-Instruct-v0.2` via HuggingFace Hub (free!)  
- **Chain**: `RetrievalQA` (stuff documents into prompt)  
- **UI**: Gradio with a file-upload widget

Run the cells below and then open the printed URL in your browser.

In [ ]:
# ─── Task 6a: Configure the LLM (Free – HuggingFace Hub) ────────────────────
from langchain_community.llms import HuggingFaceHub

# Free HuggingFace Inference API  – same Mistral family as the assignment
# Only requires a FREE HuggingFace account token (already loaded in HF_TOKEN)
llm = HuggingFaceHub(
    repo_id                  = "mistralai/Mistral-7B-Instruct-v0.2",
    huggingfacehub_api_token = HF_TOKEN,
    model_kwargs = {
        "max_new_tokens" : 512,
        "temperature"    : 0.7,
        "top_p"          : 0.95,
        "do_sample"      : True,
    },
)

print("✅  LLM configured: mistralai/Mistral-7B-Instruct-v0.2")
print("   Provider        : HuggingFace Hub (FREE tier)")
print("   Max new tokens  : 512")
print("   Temperature     : 0.7")


In [ ]:
# ─── Task 6b: Build the RetrievalQA Chain ─────────────────────────────────────
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

PROMPT_TEMPLATE = """
You are a helpful research assistant for Quest Analytics.
Use ONLY the following context excerpts from the document to answer the question.
If the answer is not contained in the context, say "I don't have enough information in the provided document to answer that."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    input_variables = ["context", "question"],
    template        = PROMPT_TEMPLATE,
)

qa_chain = RetrievalQA.from_chain_type(
    llm                  = llm,
    chain_type           = "stuff",
    retriever            = retriever,
    return_source_documents = True,
    chain_type_kwargs    = {"prompt": prompt},
)

print("✅  RetrievalQA chain built")

# Quick test
test_result = qa_chain({"query": "What this paper is talking about?"})
print()
print("── Test Query ──")
print("Q:", "What this paper is talking about?")
print()
print("A:", test_result["result"])

In [ ]:
# ─── Task 6c: Gradio QA Bot Interface ─────────────────────────────────────────
# ⬤  SCREENSHOT THIS CELL'S OUTPUT as QA_bot.png
#    Make sure to:
#      1. Upload sample_paper.pdf via the file widget
#      2. Type: What this paper is talking about?
#      3. Take screenshot of the running interface

import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
import tempfile, os

# Reuse the already-configured embeddings, llm, prompt from previous cells

def build_qa_chain(pdf_path: str):
    """(Re)builds the RAG pipeline for a newly uploaded PDF."""
    loader = PyPDFLoader(pdf_path)
    pages  = loader.load()
    splits = text_splitter.split_documents(pages)
    vdb    = Chroma.from_documents(splits, embedding=embeddings)
    ret    = vdb.as_retriever(search_type="similarity", search_kwargs={"k": 3})
    chain  = RetrievalQA.from_chain_type(
        llm=llm, chain_type="stuff", retriever=ret,
        return_source_documents=True,
        chain_type_kwargs={"prompt": prompt},
    )
    return chain

# State shared across Gradio callbacks
current_chain = {"chain": qa_chain}   # default: use the chain already built above

def upload_pdf(file):
    if file is None:
        return "⚠️  Please upload a PDF file."
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
            tmp.write(open(file.name, "rb").read())
            tmp_path = tmp.name
        current_chain["chain"] = build_qa_chain(tmp_path)
        os.unlink(tmp_path)
        return "✅  PDF uploaded and indexed successfully! Ask me anything."
    except Exception as e:
        return f"❌  Error processing PDF: {str(e)}"

def answer_question(message, history):
    if not message.strip():
        return ""
    try:
        result  = current_chain["chain"]({"query": message})
        answer  = result["result"].strip()
        sources = result.get("source_documents", [])
        if sources:
            pages_used = list({doc.metadata.get('page', '?') for doc in sources})
            answer += f"\n\n📄 *Source pages: {pages_used}*"
        return answer
    except Exception as e:
        return f"❌  Error: {str(e)}"

# ─── Build Gradio UI ──────────────────────────────────────────────────────────
with gr.Blocks(
    title="AI RAG Assistant – Quest Analytics",
    theme=gr.themes.Soft(primary_hue="blue")
) as demo:

    gr.Markdown("""
    # 🔬 AI RAG Assistant – Quest Analytics
    **Powered by LangChain + HuggingFace (mistralai/Mistral-7B-Instruct-v0.2)**

    Upload a research paper PDF and ask any question about it.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input     = gr.File(label="📁 Upload PDF", file_types=[".pdf"])
            upload_btn    = gr.Button("📤 Index PDF", variant="primary")
            upload_status = gr.Textbox(
                label="Status",
                value="Upload a PDF to begin.",
                interactive=False,
                lines=2,
            )

        with gr.Column(scale=2):
            chatbot = gr.ChatInterface(
                fn      = answer_question,
                chatbot = gr.Chatbot(height=480, label="Chat with your Document"),
                textbox = gr.Textbox(
                    placeholder='e.g. "What this paper is talking about?"',
                    label="Your Question",
                    lines=2,
                ),
                title      = "",
                retry_btn  = None,
                undo_btn   = "↩ Undo",
                clear_btn  = "🗑 Clear",
            )

    upload_btn.click(fn=upload_pdf, inputs=pdf_input, outputs=upload_status)

demo.launch(share=True)   # prints a public URL you can open in your browser


---
## ✅ All 6 Tasks Complete!

| Task | Screenshot to capture |
|------|----------------------|
| Task 1 – PDF Loader | Screenshot the **Task 1** cell (code + output) → `pdf_loader.png` |
| Task 2 – Text Splitter | Screenshot the **Task 2** cell → `code_splitter.png` |
| Task 3 – Embeddings | Screenshot the **Task 3** cell → `embedding.png` |
| Task 4 – Chroma VectorDB | Screenshot the **Task 4** cell → `vectordb.png` |
| Task 5 – Retriever | Screenshot the **Task 5** cell → `retriever.png` |
| Task 6 – QA Bot UI | Screenshot the **Gradio browser window** (with PDF uploaded + query answered) → `QA_bot.png` |